# Independent Components Analysis — Model and Density Transformations

## Learning Objectives
- Formulate the **cocktail party problem**: recover independent sources $s$ from mixed observations $x = As$
- Understand the two fundamental **ICA ambiguities**: permutation and scaling
- Prove that ICA fails for **Gaussian sources** (rotational symmetry)
- Derive the **density transformation formula**: $p_x(x) = p_s(Wx)\cdot|W|$
- Understand the role of the Jacobian determinant in change-of-variables

## Problem Statement

**Cocktail party problem:** $n$ speakers emit signals $s \in \mathbb{R}^n$; $n$ microphones record mixtures $x = As$, where $A \in \mathbb{R}^{n \times n}$ is an unknown mixing matrix.
Goal: recover $s^{(i)}$ from $x^{(i)}$ by learning the **unmixing matrix** $W = A^{-1}$.

$$s^{(i)} = W x^{(i)}, \qquad w_j^T = \text{row } j \text{ of } W$$

**Ambiguities (cannot be resolved from data alone):**
- **Permutation:** $W$ and $PW$ are indistinguishable for any permutation matrix $P$
- **Scaling:** $W$ and $\text{diag}(\alpha_j)\cdot W$ are indistinguishable (affects amplitude only)

**Why Gaussians fail.** If $s \sim \mathcal{N}(0, I)$, then $x = As \sim \mathcal{N}(0, AA^T)$.
For any orthogonal $R$, $A' = AR$ also gives $x \sim \mathcal{N}(0, AA^T)$ — so $A$ and $AR$ are indistinguishable.

**Density transformation.** If $x = W^{-1}s$ (or $s = Wx$):
$$p_x(x) = p_s(Wx)\cdot|W|$$
where $|W| = |\det W|$ is the absolute Jacobian determinant.

## 1. The Cocktail Party Problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 2000
t = np.linspace(0, 10, m)

# Two independent sources: sine wave and sawtooth
s1 = np.sin(2 * np.pi * 1.1 * t)
s2 = (t % 1.0) * 2 - 1                 # sawtooth
S  = np.column_stack([s1, s2])           # (m, 2)

# Unknown mixing matrix
A = np.array([[0.8, 0.5],
              [0.3, 0.9]])
X = S @ A.T   # x^(i) = A s^(i)

fig, axes = plt.subplots(3, 2, figsize=(14, 9))

for j, (sig, title, color) in enumerate([
    (S[:, 0], 'Source 1: sine wave', '#2166ac'),
    (S[:, 1], 'Source 2: sawtooth',  '#d6604d')
]):
    axes[0, j].plot(t[:200], sig[:200], color=color, lw=1.5)
    axes[0, j].set_title(title, fontsize=10)
    axes[0, j].set_ylabel('Amplitude')

for j, (sig, title) in enumerate([
    (X[:, 0], 'Microphone 1 (mixed)'),
    (X[:, 1], 'Microphone 2 (mixed)')
]):
    axes[1, j].plot(t[:200], sig[:200], color='gray', lw=1.5)
    axes[1, j].set_title(title, fontsize=10)
    axes[1, j].set_ylabel('Amplitude')

# Scatter: source space vs observed space
axes[2, 0].scatter(S[:, 0], S[:, 1], s=3, alpha=0.3, c='steelblue')
axes[2, 0].set_xlabel('$s_1$'); axes[2, 0].set_ylabel('$s_2$')
axes[2, 0].set_title('Source space $s$\n(independent — rectangular shape)')

axes[2, 1].scatter(X[:, 0], X[:, 1], s=3, alpha=0.3, c='gray')
axes[2, 1].set_xlabel('$x_1$'); axes[2, 1].set_ylabel('$x_2$')
axes[2, 1].set_title('Observed space $x = As$\n(mixed — parallelogram)')

fig.suptitle('Cocktail Party Problem: Independent Sources Mixed by Unknown $A$',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True mixing matrix A:\n{A}')
print(f'True unmixing matrix W = A⁻¹:\n{np.linalg.inv(A).round(4)}')

## 2. Why Gaussian Sources Break ICA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
m = 2000

# Mixing matrix
A = np.array([[1.0, 0.5], [0.3, 1.0]])

# Case 1: Non-Gaussian sources (uniform)
s_ng = np.random.uniform(-1, 1, (m, 2))
x_ng = s_ng @ A.T

# Case 2: Gaussian sources
s_g  = np.random.randn(m, 2)
x_g  = s_g @ A.T

# Alternative mixing A' = AR for orthogonal R
theta = np.pi / 5
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
Ap = A @ R
x_gp = s_g @ Ap.T   # same distribution as x_g!

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Non-Gaussian: distinct shapes in source vs observed space
axes[0, 0].scatter(s_ng[:, 0], s_ng[:, 1], s=4, alpha=0.3, c='#2166ac')
axes[0, 0].set_title('Non-Gaussian sources\n(uniform, rectangular)')
axes[0, 0].set_xlabel('$s_1$'); axes[0, 0].set_ylabel('$s_2$')

axes[0, 1].scatter(x_ng[:, 0], x_ng[:, 1], s=4, alpha=0.3, c='gray')
axes[0, 1].set_title('Observed $x = As$\n(parallelogram — shape reveals $A$)')
axes[0, 1].set_xlabel('$x_1$'); axes[0, 1].set_ylabel('$x_2$')

axes[0, 2].axis('off')
axes[0, 2].text(0.5, 0.5, 'Non-Gaussian sources:\nDistinct source geometry →\nICA can recover $A$',
                ha='center', va='center', fontsize=12,
                bbox=dict(boxstyle='round', fc='#d4edda', ec='green', lw=2),
                transform=axes[0, 2].transAxes)

# Gaussian: rotationally symmetric
axes[1, 0].scatter(s_g[:, 0], s_g[:, 1], s=4, alpha=0.3, c='#d6604d')
axes[1, 0].set_title('Gaussian sources\n(rotationally symmetric)')
axes[1, 0].set_xlabel('$s_1$'); axes[1, 0].set_ylabel('$s_2$')

axes[1, 1].scatter(x_g[:, 0],  x_g[:, 1],  s=4, alpha=0.25, c='gray', label='$A$')
axes[1, 1].scatter(x_gp[:, 0], x_gp[:, 1], s=4, alpha=0.25, c='purple', label="$A'=AR$")
axes[1, 1].set_title("$x \\sim A$ and $x \\sim A'=AR$ look identical\n(same covariance $AA^T$)")
axes[1, 1].set_xlabel('$x_1$'); axes[1, 1].set_ylabel('$x_2$')
axes[1, 1].legend(fontsize=9)

axes[1, 2].axis('off')
axes[1, 2].text(0.5, 0.5, 'Gaussian sources:\n$A$ and $AR$ indistinguishable\n→ ICA fails!\n\n'
                'Sources must be non-Gaussian\nfor ICA to work.',
                ha='center', va='center', fontsize=12,
                bbox=dict(boxstyle='round', fc='#f8d7da', ec='red', lw=2),
                transform=axes[1, 2].transAxes)

fig.suptitle('ICA Fails for Gaussian Sources: Rotational Ambiguity',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Density Transformation: $p_x(x) = p_s(Wx)\cdot|W|$

If $s = Wx$ (linear mapping with Jacobian $W$), then by the change-of-variables formula:
$$p_x(x) = p_s(Wx)\cdot|\det W|$$

The $|\det W|$ factor accounts for the **volume scaling** of the transformation.
Intuitively: if $W$ stretches volume by $|\det W|$, probability mass must be compressed by the same factor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform

np.random.seed(5)

# 1D example: s ~ Uniform[0,1], x = s/W  (scalar W)
# p_x(x) = p_s(Wx) * |W|

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

W_vals = [0.5, 1.0, 2.0]
s_samp = np.random.uniform(0, 1, 5000)

for ax, W_scalar in zip(axes, W_vals):
    # x = s / W  (inverse: s = W*x)
    x_samp = s_samp / W_scalar

    ax.hist(x_samp, bins=50, density=True, alpha=0.5, color='steelblue', label='Empirical $p_x$')

    # Theoretical: p_x(x) = p_s(Wx) * |W|
    # p_s(s) = 1{0 ≤ s ≤ 1}
    # p_x(x) = |W| * 1{0 ≤ Wx ≤ 1} = |W| * 1{0 ≤ x ≤ 1/W}
    x_vals = np.linspace(-0.1, 1.0/W_scalar + 0.2, 300)
    p_theory = np.where((x_vals >= 0) & (x_vals <= 1/W_scalar), W_scalar, 0.0)
    ax.plot(x_vals, p_theory, 'r-', lw=2.5, label=f'Theory: $p_x = |W|={{|{W_scalar}|}}$')

    ax.set_xlabel('$x$'); ax.set_ylabel('Density')
    ax.set_title(f'$W={W_scalar}$, $x = s/W$\n$p_x(x) = p_s(Wx)\\cdot|W|={W_scalar}$')
    ax.legend(fontsize=8.5)

fig.suptitle('Density Transformation: $p_x(x) = p_s(Wx)\\cdot|W|$\n'
             '(Larger $|W|$ = more compression = taller, narrower density)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 2D Density Transformation and the Jacobian

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)
m = 4000

# s ~ Uniform[-1,1]^2 (independent)
S_2d = np.random.uniform(-1, 1, (m, 2))

# Three mixing matrices with different |det A|
matrices = [
    ('$|\\det A|=1$',  np.array([[1.0, 0.5], [0.0, 1.0]])),
    ('$|\\det A|=2$',  np.array([[2.0, 0.0], [0.0, 1.0]])),
    ('$|\\det A|=0.5$', np.array([[0.5, 0.0], [0.0, 1.0]])),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for j, (label, A_mat) in enumerate(matrices):
    X_2d = S_2d @ A_mat.T
    det  = abs(np.linalg.det(A_mat))
    W_mat = np.linalg.inv(A_mat)
    detW  = abs(np.linalg.det(W_mat))

    # Source scatter
    axes[0, j].scatter(S_2d[:, 0], S_2d[:, 1], s=4, alpha=0.3, c='#2166ac')
    axes[0, j].set_xlim(-3, 3); axes[0, j].set_ylim(-3, 3)
    axes[0, j].set_aspect('equal')
    axes[0, j].set_xlabel('$s_1$'); axes[0, j].set_ylabel('$s_2$')
    axes[0, j].set_title(f'Source $s$ — unit square\n{label}')
    axes[0, j].add_patch(plt.Rectangle((-1,-1), 2, 2, fc='none', ec='red', lw=2, ls='--'))

    # Observed scatter
    axes[1, j].scatter(X_2d[:, 0], X_2d[:, 1], s=4, alpha=0.3, c='gray')
    axes[1, j].set_xlim(-3, 3); axes[1, j].set_ylim(-3, 3)
    axes[1, j].set_aspect('equal')
    axes[1, j].set_xlabel('$x_1$'); axes[1, j].set_ylabel('$x_2$')
    axes[1, j].set_title(f'Observed $x = As$\n'
                          f'$p_x(x) = p_s(Wx)\\cdot|W|$, $|W|=1/|A|={1/det:.2f}$')
    axes[1, j].text(0.03, 0.97,
                    f'Volume scaled by {det:.1f}\n'
                    f'→ density scaled by {1/det:.2f}',
                    transform=axes[1, j].transAxes, fontsize=8.5, va='top',
                    bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('2D Density Transformation: Volume Change = Inverse Density Change',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Mixed signals</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">x = As</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >x &#x2208; &#x211d;&#x207f; observed; s &#x2208; &#x211d;&#x207f; independent sources; A unknown mixing matrix</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >assume sources s&#x2c7; are independent, non-Gaussian</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Independence</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">assumption</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >p(s) = &#x220f;&#x2c7; p&#x2c7;(s&#x2c7;) &#x2014; product of marginals (no Gaussian)</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >density transformation theorem</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Density</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">transform</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >p&#x2093;(x) = p&#x209b;(Wx) &#xb7; |det W| where W = A&#x207b;&#xb9; = unmixing matrix</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >W needed to recover sources</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Recover</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">sources</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >W = A&#x207b;&#xb9;; sources s = Wx; ICA learns W from x only</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Mixing model | $x = As$ ($A$ unknown, $s$ independent non-Gaussian sources) | Cocktail-party problem |
| Unmixing matrix | $W = A^{-1}$ (what ICA learns) | $s = Wx$ recovers sources |
| Density transformation | $p_x(x) = p_s(Wx) \cdot \lvert\det W\rvert$ | Change-of-variables from $s$ to $x$ |
| Independence assumption | $p(s) = \prod_j p_j(s_j)$ | Sources are statistically independent |
| Why not Gaussian | Gaussian sources → $A$ is unidentifiable up to rotation | Need non-Gaussian source densities |

**Key insight:** ICA recovers independent sources from mixed observations by exploiting non-Gaussianity; the density transformation formula $p_x(x) = p_s(Wx)\cdot\lvert W\rvert$ connects the observed density to the source density through the unmixing matrix.